# Observables: Diagnosing Phases in Translation-Invariant ED

## Motivation and Scope

Symmetry-resolved exact diagonalisation gives us energy eigenvalues and eigenvectors labelled by symmetry quantum numbers.  The next question is: **what do we measure to identify the phase?**

For translation-invariant Hamiltonians with periodic boundary conditions, many naive diagnostics fail for fundamental, not numerical, reasons.  This notebook explains why, and provides the correct tools for distinguishing:

- **Fractional Chern insulators** (FCI)
- **Charge-density waves / Wigner crystals** (CDW)
- **Anomalous Hall crystals** (AHC)
- **Superfluids** (SF), including those condensing at specific crystal momenta
- **Supersolids** (SS)

## Density and Translation Symmetry

### What translation symmetry constrains

In a translation eigenstate, $\langle n_{R,\alpha}\rangle$ is independent of
unit cell $R$ for each sublattice $\alpha$. Different sublattices can have
different occupations. If translations act transitively on all sites, the
occupation is $N_e/N$ everywhere.

A uniform one-point density therefore does not rule out charge order in a
finite symmetric eigenstate. Connected density correlations and their size
scaling reveal ordering tendencies. Degenerate eigenstates may be returned as
arbitrary linear combinations, so density patterns should be interpreted
together with symmetry and the chosen subspace.

### Choosing a state or manifold

A translation sector fixes momentum but does not by itself identify a phase.
A full identity-symmetry solve includes all states and may choose a mixture
within a degenerate eigenspace. To compare observables consistently, specify
an energy state or a normalized projector over an isolated manifold.

The repository's full-ED density helper currently requires open boundaries.
This is an API restriction; local density remains a meaningful observable
under periodic boundaries, subject to the symmetry constraints above.

## The Correct Diagnostic Toolkit

### Overview

| Phase | Many-body Chern | $S(\bm q)$ (static structure factor) | Superfluid stiffness $\rho_s$ | $\rho(\bm k)$ (ODLRO) | Low-energy spectrum |
|-------|:---:|:---:|:---:|:---:|:---:|
| **FCI** | integer manifold C; fractional response C/m | smooth, no Bragg peaks | $\approx 0$ | broad, $O(1)$ | topological GSD (e.g. 2, 3) |
| **CDW / Wigner crystal** | 0 | **sharp Bragg peaks** at $\bm Q$ | $\approx 0$ | broad, $O(1)$ | tower of states (spacing $\bm Q$) |
| **Anomalous Hall crystal** | integer | sharp Bragg peaks at $\bm Q$ | $\approx 0$ | broad, $O(1)$ | topological GSD + tower |
| **SF @ k-point** | requires an isolated manifold | smooth, no Bragg peaks | **finite** | **macroscopic peak** at $\bm k^*$ | gapless (hard in finite size) |
| **Supersolid** | requires an isolated manifold | sharp Bragg peaks | **finite** | **macroscopic peak** at $\bm k^*$ | tower + gapless |

### 1. Static Structure Factor $S(\bm q)$ — diagnosing charge order

The connected density-density correlation function in momentum space:

\begin{equation}
    \boxed{S^{\alpha\beta}(\bm q) = \frac{1}{N}\sum_{i,j} e^{i\bm q\cdot(\bm r_i-\bm r_j)}\,\big(\langle n_i^\alpha n_j^\beta\rangle - \langle n_i^\alpha\rangle\langle n_j^\beta\rangle\big)}.
\end{equation}

$\alpha,\beta$ are optional flavour labels (e.g., sublattice A vs B).  The default `flavor_a = flavor_b = i -> true` gives the total density structure factor.

**Key properties:**
- Unlike $\langle n_i\rangle$, the **connected** correlator $\langle n_i n_j\rangle - \langle n_i\rangle\langle n_j\rangle$ carries the genuine two-point charge correlations even in a single translation sector.
- A CDW/Wigner crystal shows **sharp Bragg peaks** at the ordering wavevector $\bm Q$ and its harmonics.
- An FCI or superfluid shows a **smooth, featureless** $S(\bm q)$.

**Implementation:** `static_structure_factor(model, sector_label; ...)` computes $S(\bm q)$ at all allowed BZ momenta.  `compute_structure_factor_map(...)` scans a dense $\bm k$-grid in $[-1.5\pi, 1.5\pi]^2$ and returns data suitable for `CairoMakie.heatmap`.

**Relationship to magnetoroton physics:** In FQH/FCI systems, the magnetoroton mode softens at a finite wavevector $\bm q^*$ as the system approaches a CDW transition.  The static $S(\bm q)$ peaks at exactly this $\bm q^*$, and when the roton gap closes, $S(\bm q^*)$ diverges — this $\bm q^*$ **is** the CDW ordering wavevector $\bm Q$.

### 2. Off-Diagonal Long-Range Order $\rho(\bm k)$ — diagnosing superfluidity

The one-body density matrix and its Fourier transform:

\begin{equation}
    \boxed{\rho_{ij} = \langle a_i^\dagger a_j\rangle},\qquad
    \boxed{\rho(\bm k) = \frac{1}{N}\sum_{i,j} e^{i\bm k\cdot(\bm r_i-\bm r_j)}\,\langle a_i^\dagger a_j\rangle}.
\end{equation}

**Key properties:**
- The eigenvalues of $\rho_{ij}$ are the **natural orbital occupations**.  A superfluid has one (or a few) macroscopic eigenvalues $\sim O(N_e)$ — this is the Penrose-Onsager criterion for Bose condensation.
- $\rho(\bm k)$ directly shows **where** in the Brillouin zone the condensation occurs: a sharp peak at $\bm k^*$ with weight $\sim O(N_e)$.
- For an FCI or CDW insulator, $\rho(\bm k)$ is broad with all values $\sim O(1)$.
- For hard-core bosons, $\rho_{ij}$ is pure real-space: $a_i^\dagger a_j$ hops a particle from $j$ to $i$.  The ED implementation applies every valid hopping move and accumulates $\langle\psi|a_i^\dagger a_j|\psi\rangle$.

**Implementation:** `off_diagonal_long_range_order(model, sector_label; ...)` returns the full $N\times N$ complex matrix $\rho_{ij}$.  `compute_odlro_map(...)` performs the Fourier transform on a dense $\bm k$-grid.

**Symmetry-basis evaluation:** In a translation sector, a fixed local operator $a_i^\dagger a_j$ is not itself a symmetry-invariant Hamiltonian term.  Therefore one cannot apply it only to the orbit representative and reuse the Hamiltonian matrix-element shortcut.  The correct normalized orbit basis is
\begin{equation}
    |[s];\chi\rangle = \frac{1}{\sqrt{|G|\,|\mathrm{Stab}(s)|}}
    \sum_{g\in G}\chi(g)^*\,U_g|s\rangle .
\end{equation}
The implementation loops over every ket orbit member $U_g|s\rangle$, applies $a_i^\dagger a_j$ to that raw Fock configuration, canonicalizes the scattered mask, and obtains the coherent bra coefficient from `project_to_sector`.  This is algebraically identical to expanding the eigenvector back to the full real-space Fock basis, but it avoids materializing the full amplitude dictionary.

**Performance caution:** This exact orbit-basis contraction removes the dominant memory bottleneck of full-Fock expansion.  It does not reduce the formal all-to-all one-body workload: constructing the full $\rho_{ij}$ still scales like $N_{\mathrm{orbit}}\,|G|\,N_e\,N$ for hard-core particles.  For large clusters this is much more memory-stable, but it can still be CPU-heavy.

**Important:** $\rho(\bm k)$ and the Fourier transform of $\langle n_i\rangle$ are **completely different objects**.  $\rho(\bm k)$ measures off-diagonal (phase) coherence; $\mathcal{F}[\langle n_i\rangle]$ measures diagonal (density) modulation.  Only $\rho(\bm k)$ can detect superfluidity.

### Many-Body Chern Number and Charge Pump

For an isolated $m$-state manifold on the twist torus, the bundle Chern number
is the integer

$$C=\frac{1}{2\pi}\int_0^{2\pi}d\phi_x\int_0^{2\pi}d\phi_y\,
\mathrm{Tr}\,F_{xy},\qquad \phi_\mu=2\pi\theta_\mu.$$

`many_body_chern_number(model, :all; manifold_size=m, ...)` implements
non-Abelian determinant links and sums their plaquette phases. This is the
[Fukui–Hatsugai–Suzuki method](https://arxiv.org/abs/cond-mat/0503172).
For an appropriate equally weighted topological multiplet, a fractional Hall
response is associated with $C/m$; the manifold Chern number itself is integer.

`flux_charge_pump` follows projected Resta polarization along one flux
coordinate and returns $Q_b(\theta)=P_b(\theta)-P_b(0)$. It provides a finite-size
charge-pump diagnostic, not a fractional Chern number assigned to each branch.
Both methods reselect the lowest manifold from all momentum sectors and
check its isolation. Pump additionally checks projected-position singular
values; Chern checks overlap determinants. Neither a closed gap nor a
singular projection should be turned into a topological result.

The phase-exploration jobs use 17 points over a full $2\pi$ cycle for flow and
pump. A direct Chern calculation uses a separate two-dimensional grid.
See [charge_pump.ipynb](charge_pump.ipynb) for runnable examples.

### 4. Superfluid Stiffness (not yet implemented)

The superfluid stiffness measures the energy cost of a phase twist:

\begin{equation}
    \rho_s = \left.\frac{\partial^2 E_0(\bm\theta)}{\partial\theta_\mu^2}\right|_{\bm\theta=0}.
\end{equation}

- Finite $\rho_s$ signals superfluidity (or supersolid).
- $\rho_s \approx 0$ signals an insulator (FCI or CDW).

This can be implemented by extending the existing flux-insertion infrastructure in `spectrum_flow.jl`.

### 5. Low-Energy Spectrum — tower of states vs topological degeneracy

The pattern of nearly-degenerate low-lying energy levels differs between phases:

- **FCI:** Topological ground-state degeneracy (GSD) — e.g. 2 states for bosonic ν=1/2 Laughlin, 3 for fermionic ν=2/3.  The GSD states appear at **specific** crystal momenta determined by the anyon statistics and the torus geometry.

- **CDW / Wigner crystal:** "Tower of states" — $d$ nearly-degenerate levels (where $d$ is the number of classically degenerate CDW patterns, i.e., the period) that collapse to exact degeneracy in the thermodynamic limit.  These appear at momentum sectors spaced by the ordering wavevector $\bm Q$:
  \begin{equation}
      \bm k_0, \; \bm k_0 + \bm Q, \; \bm k_0 + 2\bm Q, \; \ldots, \; \bm k_0 + (d-1)\bm Q.
  \end{equation}

- **SF:** No robust gap — the system is gapless in the thermodynamic limit.  Finite-size gaps are spurious and scale as $1/L$.

The momentum spacing between nearly-degenerate levels is itself a diagnostic: the spacing $\Delta\bm k = \bm Q$ is the ordering wavevector.

## ⚠️  Practical Cautions

### 1. $S(\bm q)$ vs $I(q) = |\sum_i e^{i\bm q\cdot\bm r_i}\langle n_i\rangle|^2$

The trivial Fourier transform of $\langle n_i\rangle$ is **not** a substitute for $S(\bm q)$.  Since $\langle n_i\rangle$ is uniform, $I(\bm q) = \delta_{\bm q,0}\,N_e^2/N$ — only the $\bm q=0$ component is non-zero.  This contains **no information** about charge order.

### 2. $S(\bm q)$ vs $n(\bm k)$ (momentum distribution)

$S(\bm q)$ measures **diagonal** (density-density) correlations.  $n(\bm k)$ — the Fourier transform of $\langle a_i^\dagger a_j\rangle$ — measures **off-diagonal** (single-particle) correlations.  They diagnose different physics:
- $S(\bm q)$: charge order (CDW $\leftrightarrow$ FCI distinction)
- $n(\bm k) \equiv \rho(\bm k)$: superfluidity (SF $\leftrightarrow$ insulator distinction)

### 3. Choice of symmetry sector

Both $S(\bm q)$ and $\rho(\bm k)$ are well-defined within a **single** translation sector.  $\rho(\bm k)$ may show the condensation peak directly.  For $S(\bm q)$, the connected correlator $\langle n_i n_j\rangle_c$ is non-trivial even in a single sector — the CDW information is encoded in **two-point** correlations, not one-point.

### 4. Disc geometry

On a disc with open boundaries, spontaneous translational symmetry breaking **can** be observed directly in $\langle n_i\rangle$.  The current `vertices_occupation_distribution_full_ed` API requires open boundaries.  However, disc ED has strong edge effects and is limited to very small system sizes.

## API Reference

### Static Structure Factor

```julia
# At discrete BZ momenta
qs, S_q = static_structure_factor(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    flavor_a = i -> true,
    flavor_b = i -> true,
    ed_mode = :matrix,
    ed_data = nothing,
)

# Dense k-grid heatmap
kx, ky, S_map = compute_structure_factor_map(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    flavor_a = i -> true,
    flavor_b = i -> true,
    k_resolution = 61,
    ed_data = nothing,
)
heatmap(kx, ky, S_map)  # CairoMakie
```

### Off-Diagonal Long-Range Order

```julia
# Full N×N one-body density matrix
ρ = off_diagonal_long_range_order(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    ed_mode = :matrix,
    ed_data = nothing,
)

# Dense k-grid heatmap
kx, ky, ρ_map = compute_odlro_map(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    k_resolution = 61,
    ed_data = nothing,
)
heatmap(kx, ky, ρ_map)  # CairoMakie
```

### Occupation Distribution (OBC only)

```julia
⟨n⟩ = vertices_occupation_distribution_full_ed(model;
    target_eigval_idx = 1,
    filling_fraction,
    flavor_filter = i -> true,
    ed_mode = :matrix,
    ed_data = nothing,
)  # ⚠️ requires open boundary conditions
```

## Summary of Key Formulas

| Quantity | Formula |
|---|---|
| One-point density | $\langle n_i\rangle$ (uniform on translation-related sites in a translation eigenstate) |
| Connected two-point correlator | $\langle n_i n_j\rangle_c = \langle n_i n_j\rangle - \langle n_i\rangle\langle n_j\rangle$ |
| Static structure factor | $S(\bm q) = \frac{1}{N}\sum_{i,j} e^{i\bm q\cdot(\bm r_i-\bm r_j)}\,\langle n_i n_j\rangle_c$ |
| One-body density matrix | $\rho_{ij} = \langle a_i^\dagger a_j\rangle$ |
| Momentum distribution | $\rho(\bm k) = \frac{1}{N}\sum_{i,j} e^{i\bm k\cdot(\bm r_i-\bm r_j)}\,\langle a_i^\dagger a_j\rangle$ |
| Many-body Chern number | $C=(2\pi)^{-1}\int d\phi_x\,d\phi_y\,\mathrm{Tr}F_{xy}$ on the twist torus |
| Superfluid stiffness | $\rho_s = \partial^2 E_0/\partial\theta^2\|_{\theta=0}$ |

The implementation files are:
- `observables/density_distribution.jl` — OBC-only real-space $\langle n_i\rangle$
- `observables/static_structure_factor.jl` — $S(\bm q)$ with BZ heatmap
- `observables/off_diagonal_long_range_order.jl` — $\rho_{ij}$ and $\rho(\bm k)$
- `observables/charge_pump.jl` — projected-polarization charge pump
- `observables/many_body_chern_number.jl` — integer Chern number of an isolated manifold
- `observables/spectrum_flow.jl` — spectral flow diagnostics